# TNMRControl Sequence Tables
 
 Use TNMR COM commands only: list sequence tables, read table values, edit values in Python, write the full table back, and optionally save.

## 1. Setup

In [ ]:
from pathlib import Path
import importlib
import sys

redstone_dir = next(p / "hardware_capture" / "rf" / "redstone" for p in (Path.cwd(), *Path.cwd().parents) if (p / "hardware_capture" / "rf" / "redstone").is_dir())
if str(redstone_dir) not in sys.path:
    sys.path.insert(0, str(redstone_dir))

import tnmrcontrol
importlib.reload(tnmrcontrol)

from tnmrcontrol import TNMRControl

## 2. Select TNMR File

In [ ]:
input_file = Path(r"C:\TNMR\data\common\20260617_extract_waveform_phase_mod_corrected_FID_edited.tnt")

assert input_file.exists(), input_file
input_file

## 3. Connect and Open

In [ ]:
control = TNMRControl()
control.connect()
control.open(input_file)

print("Opened:", control.path)
print("Application:", control.application_info().get("application_object"))

## 4. List Sequence Tables

In [ ]:
table_names = control.get_table_list()

print("Sequence tables:", len(table_names))
for name in table_names:
    print(" -", name)

## 5. Extract All Tables

In [ ]:
tables = {}
table_errors = {}

for name in table_names:
    try:
        tables[name] = control.get_table(name)
    except RuntimeError as exc:
        table_errors[name] = str(exc)

print("Tables extracted:", len(tables))
if table_errors:
    print("Tables unavailable:", len(table_errors))
    for name, error in table_errors.items():
        print(name, error)

## 6. Preview a Table

In [ ]:
if table_names:
    table_name = table_names[0]
    print("Table:", table_name)
    print(tables.get(table_name, "<not extracted>"))
else:
    print("No sequence tables reported by TNMR.")

## 7. Table Edit Helpers

In [ ]:
def split_table_text(table_text):
    text = str(table_text).strip()
    label = ""
    if "=" in text:
        label, text = [part.strip() for part in text.split("=", 1)]
    values = text.replace(",", " ").split()
    return label, values


def join_table_values(values):
    return " ".join(str(value) for value in values)


def read_table_values(name):
    label, values = split_table_text(control.get_table(name))
    return label or name, values


def write_table_values(name, values):
    control.set_table(name, join_table_values(values))
    return control.get_table(name)


def print_indexed_values(values):
    for index, value in enumerate(values):
        print(f"{index}: {value}")

## 8. Modify One Value by Index

In [ ]:
TABLE_NAME = table_names[0] if table_names else ""
VALUE_INDEX = 0
NEW_VALUE = None  # Example: "90". Keep None to preview without modifying.

if not TABLE_NAME:
    print("No table selected.")
else:
    label, values = read_table_values(TABLE_NAME)
    print("Table:", TABLE_NAME)
    print_indexed_values(values)

    if NEW_VALUE is not None:
        if VALUE_INDEX < 0 or VALUE_INDEX >= len(values):
            raise RuntimeError(f"Index out of range: {VALUE_INDEX}")
        new_values = list(values)
        new_values[VALUE_INDEX] = str(NEW_VALUE)

        print("\nBefore:", control.get_table(TABLE_NAME))
        print("After:", write_table_values(TABLE_NAME, new_values))

## 9. Refresh Extracted Tables

In [ ]:
tables = {name: control.get_table(name) for name in control.get_table_list()}

for name, value in tables.items():
    print(f"\n[{name}]")
    print(value)

## 10. Add and Remove Note
 
 Chapter 12 documents `GetTableList`, `GetTable`, and `SetTable`. These commands expose table names and table values only. They do not document add-table, remove-table, or richer table metadata commands.

## 11. Optional Save

In [ ]:
# Save in place:
# control.save()

# Safer first test: save to a copy.
# output_file = input_file.with_name(input_file.stem + "_tables" + input_file.suffix)
# control.save_as(output_file)
# print("Saved copy:", output_file)